<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 9 · Quy trình một dự án ML</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Một vòng dự án hoàn chỉnh trên 442 bệnh nhân tiểu đường: đặt bài toán, chia dữ liệu, baseline, mô hình, Pipeline, và đo test đúng một lần.</p>
</div>

### Sau notebook này bạn sẽ

- Gọi đúng tên đặc trưng, nhãn, mô hình, tham số, siêu tham số.
- Chia train, validation, test và dùng cross-validation.
- Dựng baseline và so mô hình trên validation.
- Dùng Pipeline để chuẩn hóa không rò rỉ, và nhận ra ba kiểu rò rỉ dữ liệu.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ. Bộ dữ liệu có sẵn trong scikit-learn, không cần tải</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch09-quy-trinh-du-an-ml/01-dat-bai-toan/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Đặt bài toán và nhìn dữ liệu](#trang-1)
2. [Trang 2 · Chia train, validation, test](#trang-2)
3. [Trang 3 · Baseline và mô hình đầu tiên](#trang-3)
4. [Trang 4 · Pipeline và rò rỉ dữ liệu](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Đặt bài toán và nhìn dữ liệu

Nạp và nhìn bộ dữ liệu 442 bệnh nhân.

*Đọc kèm trên web: [Trang 1 · Đặt bài toán và nhìn dữ liệu](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch09-quy-trinh-du-an-ml/01-dat-bai-toan/)*

In [ ]:
from sklearn.datasets import load_diabetes                 # bộ dữ liệu có sẵn
X, y = load_diabetes(return_X_y=True, as_frame=True, scaled=False)   # DataFrame, giữ đơn vị gốc
print(X.shape)                                              # 442 bệnh nhân, 10 đặc trưng
print(X.head(3))                                            # 3 hàng đầu

Output mong đợi:

```text
(442, 10)
    age  sex   bmi     bp     s1     s2    s3   s4      s5    s6
0  59.0  2.0  32.1  101.0  157.0   93.2  38.0  4.0  4.8598  87.0
1  48.0  1.0  21.6   87.0  183.0  103.2  70.0  3.0  3.8918  69.0
2  72.0  2.0  30.5   93.0  156.0   93.6  41.0  4.0  4.6728  85.0
```

In [ ]:
print(y.describe().round(1))                         # nhãn: mức tiến triển sau một năm
print(X[["age", "bmi", "bp"]].describe().round(2))   # ba đặc trưng dễ hiểu nhất

Output mong đợi:

```text
count    442.0
mean     152.1
std       77.1
min       25.0
25%       87.0
50%      140.5
75%      211.5
max      346.0
Name: target, dtype: float64
          age     bmi      bp
count  442.00  442.00  442.00
mean    48.52   26.38   94.65
std     13.11    4.42   13.83
min     19.00   18.00   62.00
25%     38.25   23.20   84.00
50%     50.00   25.70   93.00
75%     59.00   29.28  105.00
max     79.00   42.20  133.00
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Không có ô thiếu. Nhãn có độ lệch chuẩn 77: mốc để so. Các đặc trưng có thang đo rất khác nhau, trang 4 sẽ thấy hậu quả.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Chia train, validation, test

Chia ba tập và cross-validation.

*Đọc kèm trên web: [Trang 2 · Chia train, validation, test](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch09-quy-trinh-du-an-ml/02-chia-du-lieu/)*

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score   # chia dữ liệu và cross-validation
from sklearn.linear_model import LinearRegression                       # hồi quy tuyến tính
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)        # 20% làm test
Xtr2, Xva, ytr2, yva = train_test_split(Xtr, ytr, test_size=0.25, random_state=42) # 25% phần còn lại làm validation
print(len(Xtr2), len(Xva), len(Xte))                                    # khoảng 60/20/20
diem = cross_val_score(LinearRegression(), Xtr, ytr, cv=5, scoring="r2")  # 5 lần chia, 5 điểm R²
print(diem.round(4), round(diem.mean(), 4), round(diem.std(), 4))       # dao động giữa các lần chia

Output mong đợi:

```text
264 89 89
[0.5476 0.3612 0.5048 0.6177 0.2149] 0.4493 0.1441
```

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI DAO ĐỘNG NHIỀU VẬY?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Cùng mô hình, chỉ khác cách chia mà R² từ 0,21 tới 0,62. Một lần đo trên khoảng 70 người rất bấp bênh, nên báo cáo trung bình kèm độ lệch.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Baseline và mô hình đầu tiên

Baseline, thêm dần đặc trưng, đo test một lần.

*Đọc kèm trên web: [Trang 3 · Baseline và mô hình đầu tiên](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch09-quy-trinh-du-an-ml/03-baseline-va-mo-hinh/)*

In [ ]:
from sklearn.dummy import DummyRegressor                       # baseline
from sklearn.metrics import mean_squared_error, r2_score       # hai thước đo

def danh_gia(mo_hinh, cot=None):                               # huấn luyện trên train, chấm trên validation
    a = Xtr2 if cot is None else Xtr2[cot]                     # chọn cột cho train
    b = Xva if cot is None else Xva[cot]                       # cùng các cột cho validation
    mo_hinh.fit(a, ytr2)                                       # học
    p = mo_hinh.predict(b)                                     # đoán validation
    return round(mean_squared_error(yva, p), 1), round(r2_score(yva, p), 4)   # MSE và R²

print(danh_gia(DummyRegressor()))                              # luôn đoán trung bình
print(danh_gia(LinearRegression(), ["bmi"]))                   # một đặc trưng
print(danh_gia(LinearRegression(), ["bmi", "bp"]))             # hai đặc trưng
print(danh_gia(LinearRegression(), ["bmi", "bp", "s5"]))       # ba đặc trưng
print(danh_gia(LinearRegression()))                            # cả 10

Output mong đợi:

```text
(5721.3, -0.0443)
(3430.1, 0.3739)
(2997.2, 0.4529)
(2713.8, 0.5047)
(2620.8, 0.5216)
```

In [ ]:
m = LinearRegression().fit(Xtr2[["bmi"]], ytr2)          # mô hình một đặc trưng
print(round(m.coef_[0], 4), round(m.intercept_, 4))      # độ dốc và hệ số chặn
mo_hinh = LinearRegression().fit(Xtr, ytr)               # chọn xong: học lại trên train + validation
p = mo_hinh.predict(Xte)                                 # đo test, đúng một lần
print(round(mean_squared_error(yte, p), 1), round(r2_score(yte, p), 4), round(mean_squared_error(yte, p) ** 0.5, 1))  # MSE, R², RMSE

Output mong đợi:

```text
10.7331 -134.5023
2900.2 0.4526 53.9
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Quay lại chỉnh mô hình sau khi thấy điểm test.</b> Từ lúc này, test đã thành validation. Điểm test chỉ được đo một lần.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Pipeline và rò rỉ dữ liệu

SGD không chuẩn hóa, Pipeline, và một đặc trưng rò rỉ.

*Đọc kèm trên web: [Trang 4 · Pipeline và rò rỉ dữ liệu](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch09-quy-trinh-du-an-ml/04-pipeline-va-ro-ri/)*

In [ ]:
from sklearn.linear_model import SGDRegressor            # hồi quy học bằng SGD
from sklearn.pipeline import make_pipeline               # ghép các bước
from sklearn.preprocessing import StandardScaler         # chuẩn hóa
s = SGDRegressor(random_state=0).fit(Xtr2, ytr2)         # học thẳng trên đơn vị gốc
print(r2_score(yva, s.predict(Xva)))                     # phân kỳ
p = make_pipeline(StandardScaler(), SGDRegressor(random_state=0))   # chuẩn hóa rồi mới hồi quy
p.fit(Xtr2, ytr2)                                        # scaler học mu, sigma chỉ từ train
print(round(r2_score(yva, p.predict(Xva)), 4))           # ngang nghiệm đóng

Output mong đợi:

```text
-5.0340614999537655e+22
0.5198
```

In [ ]:
import numpy as np                                         # sinh nhiễu
rng = np.random.default_rng(0)                             # seed
Xl = X.copy()                                              # bản sao để thêm cột
Xl["xet_nghiem_sau"] = y + rng.normal(0, 10, len(y))       # cột "bí mật" chứa gần như đáp án
a, b, c_, d = train_test_split(Xl, y, test_size=0.2, random_state=42)   # chia như trước
print(round(r2_score(d, LinearRegression().fit(a, c_).predict(b)), 4))  # đẹp giả tạo

Output mong đợi:

```text
0.9826
```

<div style="border-left:6px solid #1F5F8B;background-color:#E9F1F8;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1F5F8B;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">TRỰC GIÁC</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Hỏi: ngày mô hình được dùng thật, nó có được thấy thông tin này không? Kết quả 6 tháng sau thì không. R² 0,98 ở đây là ảo.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch09-quy-trinh-du-an-ml/05-bai-tap/)*

### Bài 9.1 · Baseline trung vị

Đoán MSE của `DummyRegressor(strategy="median")` trên validation so với baseline trung bình 5721,3.
<details><summary>Xem đáp án</summary>

Tệ hơn: 6342,1. Trung bình là hằng số tốt nhất cho MSE; trung vị tốt nhất cho MAE. Nhãn lại lệch phải. Code ở ô ngay dưới.
</details>

In [ ]:
print(danh_gia(DummyRegressor(strategy="median")))          # baseline trung vị
print(round(ytr2.mean(), 2), ytr2.median(), round(yva.mean(), 2))   # trung bình train, trung vị train, trung bình validation

Output mong đợi:

```text
(6342.1, -0.1576)
149.81 136.0 165.38
```

### Bài 9.2 · Tự tính R²

Tính $1 - \text{MSE}/\text{phương sai của yva}$ cho mô hình 10 đặc trưng và baseline.
<details><summary>Xem đáp án</summary>

Phương sai 5478,8. Ra 0,5216 và $-0{,}0443$, khớp Bảng 9.2. Baseline âm vì đoán trung bình của train chứ không phải của validation. Code ở ô ngay dưới.
</details>

In [ ]:
v = ((yva - yva.mean()) ** 2).mean()                          # MSE nếu đoán đúng trung bình validation
print(round(v, 1), round(1 - 2620.8 / v, 4), round(1 - 5721.3 / v, 4))   # R² của mô hình và baseline

Output mong đợi:

```text
5478.8 0.5216 -0.0443
```

### Bài 9.3 · Tìm rò rỉ

Mô hình dự đoán khách hủy gói cước đạt 97%. Đặc trưng có "ngày hủy gói"; chuẩn hóa trên cả bộ rồi mới chia; thử 40 cấu hình rồi báo điểm test tốt nhất. Chỉ ra từng chỗ rò rỉ.
<details><summary>Xem đáp án</summary>

Ngày hủy gói là nhãn viết dạng khác (kiểu 1). Chuẩn hóa trước khi chia (kiểu 2). Chọn bằng test (kiểu 3). Thêm nữa: nếu chỉ 3% khách hủy, luôn đoán "không hủy" đã được 97%.
</details>